# CFA — Contextual Flux Architecture
## LLM-Powered Governance Demo

| Section | Feature | LLM Role |
|---|---|---|
| S1 | Setup & Secrets | Read API key from Databricks Secret Scope |
| S2 | Catalog | Operational metadata |
| S3 | LLM Normalizer | Semantic intent resolution (replaces keywords) |
| S4 | Strict Mode | LLM output validated against catalog |
| S5 | LLM Audit Trail | Every call SHA-256 traceable |
| S6 | LLM Systematizer | NL governance → BehaviorSpec → PolicyRules |
| S7 | Full Kernel + LLM | End-to-end governed execution |
| S8 | Runtime Gate + LLM | Guard with LLM-backed validation |
| S9 | Compare Normalizers | Rule-based vs LLM side-by-side |
| S10 | PolicyEngine + LLM Rules | Close the NL→Rules→Engine loop |

**API key:** stored in Databricks Secret Scope `cfa-secrets`, never in code.

---


## Section 1 — Setup & Secrets

API key read from Databricks Secret Scope `cfa-secrets/deepseek-key`.
Falls back to `DEEPSEEK_API_KEY` env var for local testing.


## Section 0 — Install

Run this cell first to install dependencies.


In [ ]:
%pip install -q cfa-kernel[all] openai


In [ ]:
# --- Test if openai is importable ---
HAS_OPENAI = False
try:
    import openai
    print(f"openai {openai.__version__} - OK")
    HAS_OPENAI = True
except ImportError as e:
    print(f"ERROR: openai not importable: {e}")
    print("Run: %%pip install openai")

import cfa, os
from cfa.normalizer.llm import OpenAILMProvider, LLMNormalizerBackend
from cfa.normalizer.base import IntentNormalizer, RuleBasedNormalizerBackend
from cfa.runtime import RuntimeGate
from cfa.policy.engine import PolicyEngine
from cfa.types import StateSignature
from cfa.core.kernel import KernelConfig

print(f"CFA version: {cfa.__version__}")

# --- Read API keys from secret scope ---
def _try_secret(key):
    try:
        return dbutils.secrets.get("cfa-secrets", key)
    except Exception:
        return None

# Try Databricks secrets, fall back to env vars
OPENAI_KEY = _try_secret("openai-key") or os.environ.get("OPENAI_API_KEY")
DEEPSEEK_KEY = _try_secret("deepseek-key") or os.environ.get("DEEPSEEK_API_KEY")

# Auto-select: prefer OpenAI (better reachability from Azure), fallback to DeepSeek
if OPENAI_KEY:
    LLM_MODEL = "gpt-4o-mini"
    LLM_BASE_URL = None  # default api.openai.com
    LLM_API_KEY = OPENAI_KEY
    print(f"Using OpenAI ({LLM_MODEL})")
elif DEEPSEEK_KEY:
    LLM_MODEL = "deepseek-chat"
    LLM_BASE_URL = "https://api.deepseek.com"
    LLM_API_KEY = DEEPSEEK_KEY
    print(f"Using DeepSeek ({LLM_MODEL})")
else:
    LLM_MODEL = None
    LLM_BASE_URL = None
    LLM_API_KEY = None
    print("WARNING: No API key found.")
    print("  Databricks: secrets create-scope cfa-secrets")
    print("  Local:      set OPENAI_API_KEY or DEEPSEEK_API_KEY env var")

# Combined gate: need both openai + API key
HAS_LLM = HAS_OPENAI and LLM_API_KEY is not None

# Helper: create an LLM provider with the auto-detected config
def make_provider():
    """Return OpenAILMProvider with auto-detected config."""
    return OpenAILMProvider(
        model=LLM_MODEL,
        api_key=LLM_API_KEY,
        base_url=LLM_BASE_URL,
        temperature=0.0,
    )


---
## Section 2 — Data Catalog

Catalog grounds the LLM. The LLM receives it in the prompt and MUST reference real datasets.


In [ ]:
CATALOG = {
    'nfe_bronze': {
            'classification': 'internal',
        'type': 'delta', 'layer': 'bronze', 'size_gb': 50,
        'partition_by': ['processing_date'], 'pii': False,
        'description': 'Notas Fiscais Eletronicas brutas',
    },
    'clientes_bronze': {
            'classification': 'sensitive',
        'type': 'delta', 'layer': 'bronze', 'size_gb': 10,
        'partition_by': ['processing_date'], 'pii': True,
        'pii_columns': ['cpf', 'nome', 'endereco'],
        'description': 'Dados cadastrais com CPF e endereco',
    },
    'vendas_bronze': {
            'classification': 'high_volume',
        'type': 'delta', 'layer': 'bronze', 'size_gb': 2000,
        'pii': False,
        'description': 'Registros de transacoes de venda',
    },
    'fornecedores_bronze': {
            'classification': 'internal',
        'type': 'delta', 'layer': 'bronze', 'size_gb': 10,
        'pii': False,
        'description': 'Cadastro de fornecedores',
    },
    'vendas_gold_agregado': {
            'classification': 'high_volume',
        'type': 'delta', 'layer': 'gold', 'size_gb': 500,
        'pii': False,
        'description': 'Agregados de vendas para BI',
    },
}

from cfa.policy.catalog import validate_catalog
result = validate_catalog(CATALOG)
print(f"Catalog valid: {result.valid}")
print(f"  {len(CATALOG)} datasets registered")
for msg in result.messages:
    print(f"  {msg}")


---
## Section 3 — LLM Normalizer: Semantic Intent Resolution

Rule-based = keyword matching (`join`, `aggregate`, `anonymize`).
LLM = sends `intent + catalog` to DeepSeek, parses structured JSON.

The `SemanticResolution` returned contains the resolved `StateSignature`.


In [ ]:
if not HAS_LLM:
    print("SKIP: no API key configured.")
else:
    provider = make_provider()
    llm_backend = LLMNormalizerBackend(provider=provider, strict=False)
    normalizer = IntentNormalizer(backend=llm_backend)

    print("=" * 55)
    print("LLM NORMALIZER -- Semantic Resolution")
    print("=" * 55)

    intents = [
        "Join NFe with Clientes anonymize CPF and persist to Silver",
        "Aggregate vendas by region persist to Gold",
        "Export raw clientes PII to Gold",
    ]

    for raw in intents:
        print(f'\nIntent: "{raw}"')
        res = normalizer.normalize(raw, {}, CATALOG)
        sig = res.signature
        print(f"  Domain       : {sig.domain}")
        print(f"  Intent       : {sig.intent}")
        print(f"  Target layer : {sig.target_layer}")
        ds_names = [d.name for d in sig.datasets]
        print(f"  Datasets     : {ds_names}")
        print(f"  Confidence   : {res.confidence_score:.2f}")
        print(f"  Ambiguity    : {res.ambiguity_level}")
        if res.reasoning:
            print(f"  Reasoning    : {res.reasoning[:150]}...")

    print("\nLLM normalizer baseline complete")


---
## Section 4 — Strict Mode: Catalog-Grounded LLM

`strict=True` validates the LLM output against the catalog.
Hallucinated datasets/references raise `ValueError`.


In [ ]:
if not HAS_LLM:
    print("SKIP: no API key available")
else:
    sp = make_provider()
    sb = LLMNormalizerBackend(provider=sp, strict=True)
    sn = IntentNormalizer(backend=sb)

    print("=" * 55)
    print("STRICT MODE — Catalog-grounded LLM")
    print("=" * 55)

    for raw in [
        "Join NFe with Clientes anonymize CPF persist to Silver",
        "Full scan vendas without partition to Gold",
    ]:
        print(f"\nIntent: \"{raw}\"")
        try:
            res = sn.normalize(raw, {}, CATALOG)
            print(f"  PASSED  | confidence={res.confidence_score:.2f}  layer={res.signature.target_layer}")
        except ValueError as e:
            print(f"  REJECTED | {str(e)[:120]}")

    print("\nStrict mode demo complete")


---
## Section 5 — LLM Audit Trail

Every LLM call is SHA-256 hashed (prompt, response, catalog).
Full traceability for compliance audits.


In [ ]:
if not HAS_LLM:
    print("SKIP: no API key available")
else:
    # The 'sb' backend from section 4 already recorded calls
    records = sb.audit_records
    print("=" * 55)
    print("LLM AUDIT TRAIL — Tamper-evident LLM calls")
    print("=" * 55)
    print(f"  Total LLM calls recorded: {len(records)}")

    for i, rec in enumerate(records):
        print(f"\n  Call {i+1}:")
        print(f"    Model         : {rec.model}")
        print(f"    Prompt hash   : {rec.prompt_hash[:16]}...")
        print(f"    Response hash : {rec.response_hash[:16]}...")
        print(f"    Catalog hash  : {rec.catalog_hash[:16]}...")
        if rec.catalog_validation_errors:
            for err in rec.catalog_validation_errors:
                print(f"    Val. error    : {err}")
        if rec.parsed_json:
            keys = list(rec.parsed_json.keys())
            print(f"    Parsed keys   : {keys[:8]}")

    print("\nLLM audit trail complete")


---
## Section 6 — LLM Systematizer: NL → PolicyRules

Describe governance in natural language. LLM produces `BehaviorSpec`.
`Systematizer` converts it to executable `PolicyRule` objects.


In [ ]:
from cfa.behavior.llm import OpenAISystematizerBackend
from cfa.behavior import Systematizer

taxonomy, rules = None, None

if not HAS_LLM:
    print("SKIP: no LLM available (missing openai or API key)")
else:
    try:
        print("=" * 55)
        print("LLM SYSTEMATIZER -- NL to Policy Rules")
        print("=" * 55)

        governance_desc = """\
Fiscal ETL pipeline governance:
1. All datasets moving from Bronze to Silver/Gold MUST anonymize PII (CPF, nome, endereco)
2. Every Silver and Gold write requires a merge_key (upsert, not append)
3. Datasets larger than 100GB MUST have partition_by defined
4. No raw PII columns may appear in Gold layer
5. Maximum cost per pipeline run: 50 DBU
"""

        sys_backend = OpenAISystematizerBackend(
            model=LLM_MODEL,
            api_key=LLM_API_KEY,
            base_url=LLM_BASE_URL,
            temperature=0.0,
            max_tokens=2048,
        )

        taxonomy, rules = Systematizer().systematize_from_nl(
            governance_desc,
            backend=sys_backend,
            context="Fiscal ETL: NFe, Clientes, Vendas, Fornecedores on Databricks Delta Lake",
            target_layer="silver",
        )

        print(f"  Categories      : {taxonomy.category_count}")
        print(f"  Rules generated : {len(rules)}")
        print()

        for rule in rules:
            print(f"  Rule : {rule.name}")
            print(f"    code     : {rule.fault_code}")
            print(f"    action   : {rule.action.value}")
            print(f"    severity : {rule.severity.value}")
            print()

        test_intents = taxonomy.generate_test_intents(3)
        print(f"  CI test intents generated: {len(test_intents)}")
        for intent in test_intents:
            print(f"    - {intent}")

        print("\nLLM Systematizer complete")
    except Exception as e:
        print(f"  LLM call failed: {type(e).__name__}: {str(e)[:100]}")


---
## Section 7 — Full Kernel with LLM Normalizer

Same 5-phase pipeline (Formalize → Govern → Generate → Execute → Validate/Audit).
Only the **Formalize** phase switches from keyword-matching to DeepSeek.


In [ ]:
from cfa import KernelOrchestrator

if not HAS_LLM:
    print("SKIP: no API key available")
else:
    provider = make_provider()
    backend = LLMNormalizerBackend(provider=provider, strict=True)

    kernel = KernelOrchestrator(
        catalog=CATALOG,
        config=KernelConfig(
            policy_bundle_version="fiscal-prod-v1.0",
            backend="pyspark",
        ),
        normalizer_backend=backend,
    )

    print("=" * 55)
    print("FULL KERNEL + LLM NORMALIZER")
    print("=" * 55)

    for intent in [
        "Join NFe with Clientes anonymize CPF and persist to Silver",
        "Aggregate vendas by region persist to Gold",
        "Export raw clientes PII to Gold",
    ]:
        print(f"\nIntent: \"{intent}\"")
        try:
            result = kernel.process(intent)
            print(f"  -> Decision  : {result.state.value.upper()}")
            if result.signature is not None:
                print(f"  -> Sig hash  : {result.signature.signature_hash[:24]}...")
            if result.replan_history:
                print(f"  -> Replans   : {len(result.replan_history)}")
            if result.generated_code and result.generated_code.code:
                lines = result.generated_code.code.splitlines()
                print(f"  -> Code gen  : {len(lines)} lines {result.generated_code.language}")
        except Exception as e:
            print(f"  -> BLOCKED   : {type(e).__name__} | {str(e)[:100]}")

    calls = backend.audit_records
    print(f"\n  LLM calls by kernel: {len(calls)}")
    for rec in calls:
        print(f"    prompt={rec.prompt_hash[:16]}...  response={rec.response_hash[:16]}...")

    print("\nFull kernel + LLM complete")


---
## Section 8 — Runtime Gate

`RuntimeGate` validates intents before execution.
With `policy_rules` from LLM Systematizer, the gate inherits NL-defined governance.


In [ ]:
import traceback
print("=" * 55)
print("RUNTIME GATE")
print("=" * 55)

try:
    gate = RuntimeGate(
        catalog=CATALOG,
        policy_rules=rules if rules is not None else None,
    )

    result = gate.validate("Join NFe with Clientes and persist to Silver")
    print(f"  validate() -> state={result.state.value}  passed={result.passed}")
    print(f"  gate_id={result.gate_id}  execution_id={result.execution_id[:8]}...")

    @gate.guard("aggregate sales data with PII protected")
    def my_pipeline():
        return "pipeline executed"

    print(f"  @gate.guard -> {my_pipeline()}")
    print("\nRuntime Gate demo complete")
except Exception as e:
    print(f"  RUNTIME GATE FAILED: {type(e).__name__}")
    traceback.print_exc()


---
## Section 9 — Comparing Normalizers: Rule-Based vs LLM

Same intent, two backends — side-by-side comparison.


In [ ]:
print("=" * 55)
print("NORMALIZER COMPARISON: Rule-Based vs LLM")
print("=" * 55)

rule_norm = IntentNormalizer(backend=RuleBasedNormalizerBackend())
llm_norm = None
if HAS_LLM:
    provider = make_provider()
    llm_norm = IntentNormalizer(backend=LLMNormalizerBackend(provider=provider, strict=True))

header = f"{'Intent':<50s} | {'Method':<12s} | {'Domain':<10s} | {'Layer':<8s} | {'Confidence':>10s}"
print(header)
print("-" * len(header))

for raw in [
    "Join NFe with Clientes anonymize CPF and persist to Silver",
    "Aggregate vendas by region persist to Gold",
    "Export raw clientes PII to Gold",
]:
    # Rule-based
    rb = rule_norm.normalize(raw, {}, CATALOG)
    sig_rb = rb.signature
    print(f"{raw[:47]:<50s} | {'rule-based':<12s} | {sig_rb.domain:<10s} | {sig_rb.target_layer:<8s} | {rb.confidence_score:>9.2f}")
    # LLM
    if llm_norm is not None:
        try:
            llm = llm_norm.normalize(raw, {}, CATALOG)
            sig_llm = llm.signature
            print(f"{'':50s} | {'LLM':<12s} | {sig_llm.domain:<10s} | {sig_llm.target_layer:<8s} | {llm.confidence_score:>9.2f}")
        except Exception as e:
            print(f"{'':50s} | {'LLM':<12s} | ERROR: {type(e).__name__}")
    else:
        print(f"{'':50s} | {'LLM':<12s} | (no API key)")
    print()

print("Normalizer comparison complete")


---
## Section 10 — PolicyEngine with LLM-Generated Rules

Rules from `LLMSystematizer` (Section 6) feed `PolicyEngine`.
Closes the loop: NL governance → LLM → BehaviorSpec → PolicyRules → Engine.


In [ ]:
from cfa.types import DatasetRef, SignatureConstraints, TargetLayer, ExecutionContext

print("=" * 55)
print("POLICY ENGINE WITH LLM-GENERATED RULES")
print("=" * 55)

if rules is None:
    print("SKIP: rules not available (LLM Systematizer requires API key)")
    print("   Using default policy rules instead.")
    engine = PolicyEngine()  # default rules
else:
    print(f"  Using {len(rules)} LLM-generated rules from Section 6")
    engine = PolicyEngine(rules=rules)

# --- Safe signature ---
sig_safe = StateSignature(
    domain="fiscal",
    intent="Join NFe with Clientes anonymize CPF persist to Silver",
    source_intent_raw="Join NFe with Clientes anonymize CPF persist to Silver",
    target_layer=TargetLayer.SILVER,
    datasets=(
        DatasetRef(name="nfe_bronze"),
        DatasetRef(name="clientes_bronze", pii_columns=("cpf", "nome")),
    ),
    constraints=SignatureConstraints(
        no_pii_raw=True,
        merge_key_required=True,
        partition_by=("processing_date",),
    ),
    execution_context=ExecutionContext(
        policy_bundle_version="v1.0",
        catalog_snapshot_version="catalog_default",
        context_registry_version_id="ctx-default",
    ),
)

r = engine.evaluate(sig_safe)
print(f"\n  Safe signature   -> {r.action.value.upper()}  faults={len(r.faults)}")

# --- Unsafe signature ---
sig_unsafe = StateSignature(
    domain="fiscal",
    intent="Export raw clientes PII to Gold",
    source_intent_raw="Export raw clientes PII to Gold",
    target_layer=TargetLayer.GOLD,
    datasets=(
        DatasetRef(name="clientes_bronze", pii_columns=("cpf", "nome", "endereco")),
    ),
    constraints=SignatureConstraints(
        no_pii_raw=False,
        merge_key_required=False,
        partition_by=(),
    ),
    execution_context=ExecutionContext(
        policy_bundle_version="v1.0",
        catalog_snapshot_version="catalog_default",
        context_registry_version_id="ctx-default",
    ),
)

r2 = engine.evaluate(sig_unsafe)
print(f"  Unsafe signature -> {r2.action.value.upper()}  faults={len(r2.faults)}")
for fault in r2.faults[:4]:
    print(f"    [{fault.severity.value.upper():<10s}] {fault.code}")

print("\nLLM-rules PolicyEngine complete")


---
## Summary

| LLM Feature | Status | Notes |
|---|---|---|
| LLM Normalizer | \\u2713 | Semantic intent resolution via DeepSeek |
| Strict Mode | \\u2713 | LLM output validated against catalog |
| LLM Audit Trail | \\u2713 | Every call SHA-256 traceable |
| LLM Systematizer | \\u2713 | NL → BehaviorSpec → PolicyRules |
| Full Kernel + LLM | \\u2713 | End-to-end with LLM normalizer |
| Runtime Gate | \\u2713 | Guard with LLM-generated policy rules |
| Normalizer Comparison | \\u2713 | Rule-based vs LLM side-by-side |
| Secret Management | \\u2713 | Databricks Secret Scope `cfa-secrets` |

**Key differences from rule-based:**

- LLM understands intent **semantics**, not just keywords
- Natural language governance descriptions → executable rules
- Every LLM call is audited (prompt/response/catalog SHA-256)
- Strict mode prevents hallucinations by validating against catalog
- API key stored in **Databricks Secret Scope** — never in code

**Secret scope setup (one-time):**
```bash
databricks secrets create-scope cfa-secrets
databricks secrets put-secret cfa-secrets deepseek-key
```

---

**Links**
- [Documentation](https://marquesantero.github.io/cfa/docs/intro)
- [PyPI](https://pypi.org/project/cfa-kernel/)
- [GitHub](https://github.com/marquesantero/cfa)
